# Daily CO₂ emissions
Explore country and sector trends, compare country clusters, and evaluate LSTM forecasts against naive baselines using the bundled Carbon Monitor data.

Run the cells in order. The default workflow uses local data; population and GDP enrichment is optional.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

ROOT = Path.cwd()
if not (ROOT / "co2_analysis").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "co2_analysis").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory")
sys.path.insert(0, str(ROOT))
from co2_analysis.data import load_data, daily_series, data_summary, AGGREGATES
from co2_analysis.forecast import fit_forecast
OUTPUT = ROOT / "results/notebook"
OUTPUT.mkdir(parents=True, exist_ok=True)
EPOCHS = int(os.environ.get("CO2_EPOCHS", "30"))
RUN_WORLD_BANK = False


## Dataset
The workbook contains 204,540 observations across six sectors and 14 geographies. Its mixed text and Excel dates need a format correction before analysis. The loader checks the repaired calendar and sector coverage; details are in `docs/methodology.md`.


In [ ]:
df = load_data()
display(data_summary(df))
df.head()


In [ ]:
df[['value']].describe()


In [ ]:
df.info()


In [ ]:
print("Number of geographies (including aggregates): " + str(df['country'].nunique()))
df['country'].unique()


In [ ]:
df['sector'].unique()


In [ ]:
df['date'].min(), df['date'].max()


## Population and GDP
This optional section uses live World Bank indicators. Install `requirements-optional.txt` and set `RUN_WORLD_BANK = True` above to run it. It uses complete years through 2024 and does not affect the forecasting results.


In [ ]:
if RUN_WORLD_BANK:
    import wbgapi as wb

    def fetch_pop_gdp(countries_iso3, years):
        """Return annual population and GDP indexed by country and year."""
        indicators = {'population': 'SP.POP.TOTL', 'gdp': 'NY.GDP.MKTP.CD'}
        tables = []
        for name, indicator in indicators.items():
            wide = wb.data.DataFrame(indicator, economy=countries_iso3,
                                    time=years, labels=False, numericTimeKeys=True)
            long = wide.stack().rename(name).reset_index()
            long.columns = ['country', 'year', name]
            long['year'] = long['year'].astype(int)
            tables.append(long)
        return tables[0].merge(tables[1], on=['country', 'year'], how='outer')

    countries = ['BRA', 'CHN', 'FRA', 'DEU', 'IND', 'ITA', 'JPN', 'RUS', 'ESP', 'GBR', 'USA']
    pop_gdp_df = fetch_pop_gdp(countries, range(2019, 2025))
    display(pop_gdp_df.head())
else:
    print('World Bank enrichment is disabled.')


In [ ]:
if RUN_WORLD_BANK:
    name_to_iso3 = {
        'Brazil': 'BRA', 'China': 'CHN', 'France': 'FRA', 'Germany': 'DEU',
        'India': 'IND', 'Italy': 'ITA', 'Japan': 'JPN', 'Russia': 'RUS',
        'Spain': 'ESP', 'United Kingdom': 'GBR', 'United States': 'USA',
    }
    df_norm = df.loc[~df.country.isin(AGGREGATES)].copy()
    df_norm['year'] = df_norm.date.dt.year
    df_norm['country_iso3'] = df_norm.country.map(name_to_iso3)
    df_norm = df_norm.merge(
        pop_gdp_df.rename(columns={'country': 'country_iso3'}),
        on=['country_iso3', 'year'], how='left', validate='many_to_one',
    ).rename(columns={'country': 'country_name'})
    df_norm['tCO2_day'] = df_norm.value * 1e6
    df_norm['tCO2_per_capita_day'] = df_norm.tCO2_day / df_norm.population
    df_norm['tCO2_per_million_USD_day'] = df_norm.tCO2_day / (df_norm.gdp / 1e6)

    # Omit the partial final year from annual comparisons.
    annual_df = (
        df_norm.loc[df_norm.year <= 2024]
        .groupby(['country_iso3', 'country_name', 'year'], as_index=False)
        .agg(MtCO2_per_year=('value', 'sum'), population=('population', 'first'),
             gdp=('gdp', 'first'))
    )
    annual_df['tCO2_per_capita_year'] = annual_df.MtCO2_per_year * 1e6 / annual_df.population
    annual_df['tCO2_per_million_USD_year'] = annual_df.MtCO2_per_year * 1e6 / (annual_df.gdp / 1e6)
    display(annual_df.head())
else:
    print('World Bank enrichment is disabled.')


In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print('\nDuplicates based on Date, Country & Sector')
df.duplicated(subset=['date', 'country', 'sector']).sum()


In [ ]:
full_range = pd.date_range(start=df['date'].min(), end=df['date'].max())
missing_dates = full_range.difference(df['date'].unique())
print(f"Missing days in the dataset: {len(missing_dates)}")


In [ ]:
missing_by_group = {}
for (country, sector), group in df.groupby(['country', 'sector']):
    dates = pd.DatetimeIndex(group['date'])
    missing = pd.date_range(dates.min(), dates.max()).difference(dates)
    if len(missing):
        missing_by_group[(country, sector)] = len(missing)
print(f"Country-sector pairs with missing dates: {len(missing_by_group)}")


In [ ]:
import matplotlib.pyplot as plt
df_world = df[df['country'] == 'WORLD']
world_daily = df_world.groupby('date', as_index=False)['value'].sum()
plt.figure(figsize=(12,5))
plt.plot(world_daily['date'], world_daily['value'], label='Daily')
plt.title('Global daily CO₂ emissions ')
plt.xlabel('Date'); plt.ylabel('Emissions (MtCO₂/day)')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
world_daily['rolling_30d'] = world_daily['value'].rolling(window=30, center=True).mean()

plt.figure(figsize=(12,5))
plt.plot(world_daily['date'], world_daily['value'], alpha=0.4, label='Daily')
plt.plot(world_daily['date'], world_daily['rolling_30d'], color='red', linewidth=2, label='30-day rolling mean')
plt.title('Global daily CO₂ emissions (smoothed)')
plt.xlabel('Date')
plt.ylabel('Emissions (MtCO₂/day)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
world_monthly = world_daily.copy()
world_monthly['date'] = pd.to_datetime(world_monthly['date'])
world_monthly = world_monthly.resample('ME', on='date')['value'].mean().reset_index()

plt.figure(figsize=(12,5))
plt.plot(world_monthly['date'], world_monthly['value'], color='orange', linewidth=2, label='Monthly mean')
plt.title('Global monthly CO₂ emissions')
plt.xlabel('Date')
plt.ylabel('Emissions (MtCO₂/day)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
world_monthly['value'].describe()


In [ ]:
world_monthly['value'].plot(kind='hist', bins=20)
plt.title('Distribution of monthly CO₂ emissions')
plt.xlabel('Emissions (MtCO₂/day)')
plt.show()


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
decomp = seasonal_decompose(world_monthly.set_index('date')['value'], model='additive', period=12)
decomp.plot()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
df_world = df[df['country'] == 'WORLD']
world_time_sector = (df_world
                     .groupby(['date', 'sector'], as_index=False)['value']
                     .sum())
plt.figure(figsize=(12,6))
sns.lineplot(data=world_time_sector, x='date', y='value', hue='sector', linewidth=1.5)
plt.title('Global CO₂ emissions by sector over time ')
plt.xlabel('Date'); plt.ylabel('Emissions (MtCO₂/day)')
plt.legend(title='Sector', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()


In [ ]:
world_sector_mean = (df_world
                     .groupby('sector', as_index=False)['value']
                     .mean()
                     .sort_values('value', ascending=False))
plt.figure(figsize=(10,5))
plt.barh(world_sector_mean['sector'], world_sector_mean['value'])
plt.gca().invert_yaxis()
plt.title('Global average daily emissions by sector')
plt.xlabel('Mean emissions (MtCO₂/day)'); plt.ylabel('Sector')
plt.tight_layout(); plt.show()


In [ ]:
# WORLD and EU27 & UK overlap with the individual country records.
shares = df.loc[~df.country.isin({'WORLD', 'EU27 & UK'})].groupby('country').value.sum()
shares = (100 * shares / shares.sum()).sort_values()
shares.plot.barh(figsize=(9, 5))
plt.title('Share of emissions across countries and ROW')
plt.xlabel('Share of total emissions (%)')
plt.ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates

def plot_country_emissions_time_series(df, countries):
    """Compare daily sector totals for the selected countries."""
    plt.figure(figsize=(14, 8))
    for country in countries:
        df_country = df[df['country'] == country].groupby('date')['value'].sum().reset_index()
        plt.plot(df_country['date'], df_country['value'], label=country, linewidth=1.5)

    plt.title('Daily CO₂ Emissions Over Time for Selected Countries', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Total Emissions (MtCO₂/day)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

selected_countries = ['United States', 'China', 'India', 'Russia', 'Japan', 'Germany']
plot_country_emissions_time_series(df, selected_countries)


In [ ]:
df_grouped_heatmap = df.groupby(['country', 'sector', 'date'])['value'].mean().reset_index()

df_pivot_heatmap = df_grouped_heatmap.pivot_table(index='country', columns='sector', values='value', aggfunc='mean')

plt.figure(figsize=(12, 8))
sns.heatmap(df_pivot_heatmap, annot=True, cmap='YlGnBu', fmt=".2f", linewidths=.5)
plt.title('Average Daily CO₂ Emissions by Country and Sector (MtCO₂/day)', fontsize=16)
plt.xlabel('Sector', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
if RUN_WORLD_BANK:
    year_to_plot = 2023
    df_plot = (annual_df[annual_df["year"] == year_to_plot]
               .dropna(subset=["tCO2_per_capita_year"])
               .sort_values("tCO2_per_capita_year", ascending=False))

    plt.figure(figsize=(10, 6))
    plt.bar(df_plot["country_name"], df_plot["tCO2_per_capita_year"])
    plt.title(f"Annual CO₂ per Capita (tCO₂/person/year) – {year_to_plot}")
    plt.ylabel("tCO₂ per person per year")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Optional World Bank enrichment skipped.")


In [ ]:
if RUN_WORLD_BANK:
    year_to_plot = 2023
    df_plot = (annual_df[annual_df["year"] == year_to_plot]
               .dropna(subset=["tCO2_per_million_USD_year"])
               .sort_values("tCO2_per_million_USD_year", ascending=False))

    plt.figure(figsize=(10, 6))
    plt.bar(df_plot["country_name"], df_plot["tCO2_per_million_USD_year"])
    plt.title(f"Annual CO₂ Intensity (tCO₂ per million USD) – {year_to_plot}")
    plt.ylabel("tCO₂ per million USD")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Optional World Bank enrichment skipped.")


In [ ]:
window_size = 30
n_bins = 5
from scipy.stats import norm
import numpy as np
from sklearn.preprocessing import StandardScaler
breakpoints = norm.ppf(np.linspace(0, 1, n_bins + 1)[1:-1])
symbol_map = {0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e'}

df['date'] = pd.to_datetime(df['date'])

df_country_grouped = df.groupby(['country', 'date'], as_index=False)['value'].sum()

sax_by_country = {}

for country in df_country_grouped['country'].unique():
    df_country = df_country_grouped[df_country_grouped['country'] == country].copy()
    df_country = df_country.set_index('date').sort_index()

    scaler = StandardScaler()
    df_country['value_norm'] = scaler.fit_transform(df_country[['value']])

    # Overlapping rolling means provide a descriptive symbolic sequence.
    rolling_means = df_country['value_norm'].rolling(window=window_size).mean().dropna()

    symbols = np.digitize(rolling_means, breakpoints)
    sax_sequence = ''.join([symbol_map[i] for i in symbols])

    sax_by_country[country] = sax_sequence

list(sax_by_country.items())[:3]


In [ ]:
selected_countries = ['United States', 'China', 'Russia', 'India', 'Spain', 'Italy']

filtered_sax = {country: seq for country, seq in sax_by_country.items() if country in selected_countries}

min_len = min(len(seq) for seq in filtered_sax.values())
trimmed_sax = {country: list(seq[:min_len]) for country, seq in filtered_sax.items()}

sax_df = pd.DataFrame(trimmed_sax).T

symbol_order = {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4}
sax_df_num = sax_df.map(lambda x: symbol_order.get(x, np.nan))

plt.figure(figsize=(16, 10))
sns.heatmap(
    sax_df_num,
    cmap='viridis',
    cbar_kws={'label': 'Symbol Value (a=0 → e=4)'}
)
plt.title("Rolling symbolic sequences (descriptive comparison)")
plt.xlabel("Daily rolling-window index")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


In [ ]:
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def ngrams(seq, n=3):
    return [seq[i:i+n] for i in range(len(seq)-n+1)]

def fit_ngram_model(seqs_dict, n=3, alpha=1.0):
    counts = Counter()
    for s in seqs_dict.values():
        counts.update(ngrams(s, n))
    N = sum(counts.values())
    V = 5 ** n  # Include unseen words in the smoothing denominator.
    probs = {g: (c + alpha) / (N + alpha * V) for g, c in counts.items()}
    default_p = alpha / (N + alpha * V)
    return probs, default_p

def dates_for_country(df_country_grouped, country):
    return (df_country_grouped[df_country_grouped['country'] == country]
            .sort_values('date')['date'].reset_index(drop=True))


In [ ]:
def detect_anomalies_for_country(country, sax_by_country, df_country_grouped,
                                 probs, default_p, window_size=30, n=3, top_k=5):

    if country not in sax_by_country or len(sax_by_country[country]) < n:
        return pd.DataFrame(columns=['country','pattern','score','start_date','end_date'])
    seq = sax_by_country[country]

    gs = ngrams(seq, n)
    p = np.array([probs.get(g, default_p) for g in gs])
    scores = -np.log(p)

    idx = np.argsort(scores)[-top_k:][::-1]

    dates = dates_for_country(df_country_grouped, country)
    # Each symbol is dated at the end of its rolling window.
    symbol_dates = dates[window_size-1 : window_size-1 + len(seq)]

    rows = []
    for i in idx:
        start_date = symbol_dates.iloc[i]
        end_date = symbol_dates.iloc[min(i + n - 1, len(symbol_dates)-1)]
        rows.append({
            'country': country,
            'pattern': gs[i],
            'score': float(scores[i]),
            'start_date': start_date,
            'end_date': end_date
        })
    return pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)


In [ ]:
def plot_country_with_anomalies(country, df_country_grouped, df_anoms):
    dfc = (df_country_grouped[df_country_grouped['country'] == country]
           .sort_values('date'))
    plt.figure(figsize=(12,5))
    plt.plot(dfc['date'], dfc['value'], label=f'{country} CO₂ series', color='black')

    for _, r in df_anoms.iterrows():
        plt.axvspan(r['start_date'], r['end_date'], color='red', alpha=0.3)

    plt.title(f"Rare symbolic patterns: {country}")
    plt.xlabel("Date"); plt.ylabel("CO₂ emissions (MtCO₂/day)")
    plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
n = 3
probs, default_p = fit_ngram_model(sax_by_country, n=n, alpha=1.0)

df_anoms_world = detect_anomalies_for_country(
    "WORLD", sax_by_country, df_country_grouped,
    probs, default_p, window_size=window_size, n=n, top_k=5
)
display(df_anoms_world)
plot_country_with_anomalies("WORLD", df_country_grouped, df_anoms_world)


In [ ]:
df_anoms_italy = detect_anomalies_for_country(
    "Italy", sax_by_country, df_country_grouped,
    probs, default_p, window_size=window_size, n=n, top_k=5
)
display(df_anoms_italy)
plot_country_with_anomalies("Italy", df_country_grouped, df_anoms_italy)


In [ ]:
from statsmodels.stats.diagnostic import breaks_cusumolsresid
from statsmodels.api import add_constant, OLS

df_world = df[df['country'] == 'WORLD'].copy()
df_world = df_world.groupby('date')['value'].sum().reset_index()
df_world['date'] = pd.to_datetime(df_world['date'])
df_world = df_world.sort_values('date').reset_index(drop=True)
df_world['time_index'] = np.arange(len(df_world))

X_world = add_constant(df_world['time_index'])
y_world = df_world['value']
model_world = OLS(y_world, X_world).fit()

cusum_stat_world, p_value_world, crit_vals_world = breaks_cusumolsresid(model_world.resid, ddof=2)

residuals_world = model_world.resid
cusum_world = (residuals_world - residuals_world.mean()).cumsum()

max_dev_index_world = cusum_world.abs().idxmax()
breakpoint_date_world = df_world.loc[max_dev_index_world, 'date']

plt.figure(figsize=(12, 5))
plt.plot(df_world['date'], cusum_world, label='CUSUM', color='blue')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.axvline(breakpoint_date_world, color='red', linestyle='--', linewidth=1.5, label='Maximum cumulative residual')
plt.title('Cumulative residuals from the WORLD linear trend')
plt.xlabel('Date')
plt.ylabel('Cumulative Sum of Residuals')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Date of maximum cumulative residual: {breakpoint_date_world.date()}")
print(f"CUSUM Statistic: {cusum_stat_world:.4f}")
print(f"p-value: {p_value_world:.4f}")
print(f"Critical Values: {crit_vals_world}")


## Country clustering
Compare the 11 individual countries, excluding WORLD, ROW and EU27 & UK. Scaling across countries on each date retains differences in emissions magnitude. Use three clusters to inspect these differences, and keep both sets of cluster labels separate from the features.


In [ ]:
country_daily = df.loc[~df.country.isin(AGGREGATES)].groupby(['country', 'date'])['value'].sum()
pivot_df = country_daily.unstack('date')
if pivot_df.isna().any().any():
    raise ValueError("Missing country-days: do not replace missing emissions with zero")
pivot_scaled = pd.DataFrame(StandardScaler().fit_transform(pivot_df),
                            index=pivot_df.index, columns=pivot_df.columns)
K = range(2, min(10, len(pivot_scaled)))
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(pivot_scaled).inertia_ for k in K]
plt.figure(figsize=(6, 4))
plt.plot(list(K), inertias, marker='o')
plt.title('K-Means inertia by cluster count')
plt.xlabel('Clusters'); plt.ylabel('Inertia'); plt.show()
kmeans_labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(pivot_scaled)


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
Z = linkage(pivot_scaled.to_numpy(), method='ward')
hierarchical_labels = fcluster(Z, t=3, criterion='maxclust')
cluster_assignments = pd.DataFrame({'kmeans': kmeans_labels, 'hierarchical': hierarchical_labels}, index=pivot_scaled.index)
plt.figure(figsize=(10, 5))
dendrogram(Z, labels=pivot_scaled.index, leaf_rotation=90)
plt.title('Country clusters from Ward linkage')
plt.tight_layout(); plt.show()
display(cluster_assignments)


In [ ]:
plt.figure(figsize=(10, 6))
for cluster in sorted(set(hierarchical_labels)):
    mean = pivot_scaled.loc[hierarchical_labels == cluster].mean()
    plt.plot(mean.index, mean.values, label=f'Cluster {cluster}')
plt.title('Average scaled emissions by hierarchical cluster')
plt.legend(); plt.xlabel('Date'); plt.ylabel('Scaled emissions'); plt.show()


In [ ]:
cluster_assignments.to_csv(OUTPUT / 'country_clusters.csv')


### Reading the exploratory results
The clusters describe this dataset; three groups are an illustrative choice. Symbolic scores flag rare patterns within the observed series. They use overlapping rolling means and have no labelled anomaly test set. Seasonal and autocorrelated residuals also limit the interpretation of CUSUM p-values. The largest cumulative residual marks a feature of the fitted trend, not a confirmed causal event.


## LSTM forecasting
Use the preceding 30 days to predict the next day. Split target dates into training, validation and test periods, fit the scaler on training observations, and use validation loss for early stopping. Compare the LSTM with yesterday's value and the value from seven days earlier on the same test dates.

These are rolling one-day-ahead predictions: each new day can use observations from earlier test days. The separate 30-day recursive extension feeds predictions back into the model and has no observed outcomes in this dataset. See `docs/methodology.md` for the exact split and model settings.


### WORLD and Russia
Training runs for up to 30 epochs. `CO2_EPOCHS=1` gives a short integration run; use the default setting to reproduce the benchmark.


In [ ]:
forecast_results = {}
for geography in ['WORLD', 'Russia']:
    series = daily_series(df, geography)
    predictions, future, metadata, history = fit_forecast(series, epochs=EPOCHS)
    forecast_results[geography] = (predictions, future, metadata)
    display(pd.DataFrame(metadata['metrics']).T)
    predictions.set_index('date')[['actual', 'lstm', 'persistence', 'seasonal_naive_7']].plot(figsize=(12, 4), title=f'{geography}: one-day-ahead test predictions')
    plt.ylabel('MtCO2/day'); plt.tight_layout(); plt.show()
    plt.figure(figsize=(8, 3))
    plt.plot(history['loss'], label='train'); plt.plot(history['val_loss'], label='validation')
    plt.xlabel('Epoch (zero-based)'); plt.ylabel('Scaled MSE'); plt.legend(); plt.show()
    predictions.to_csv(OUTPUT / f'{geography.lower()}_test.csv', index=False)
    future.to_csv(OUTPUT / f'{geography.lower()}_future.csv', index=False)


### Other geographies
Run `python -m co2_analysis --countries "United States" China --epochs 30` from the repository root to use the same protocol for other countries. Interpret errors in the context of each country's emissions scale.


In [ ]:
comparison = pd.DataFrame([
    {'geography': geography, 'model': model, **scores}
    for geography, (_, _, metadata) in forecast_results.items()
    for model, scores in metadata['metrics'].items()
])
display(comparison)
comparison.to_csv(OUTPUT / 'metrics.csv', index=False)


In [ ]:
for geography, (_, future, _) in forecast_results.items():
    historical = daily_series(df, geography).iloc[-60:]
    plt.figure(figsize=(10, 4))
    plt.plot(historical.index, historical.values, label='Observed')
    plt.plot(future.date, future.lstm, 'o--', label='Recursive extension (unvalidated)')
    plt.title(f'{geography}: 30-day illustrative extension')
    plt.xlabel('Date'); plt.ylabel('MtCO2/day'); plt.legend(); plt.tight_layout(); plt.show()
